# A first local LLM experiment

This tutorial takes you from a cached open-weight model to a response, an editable
prompt, validated JSON, and a captured activation. An optional final section runs
a tiny LoRA training experiment. Coding is just one application of the same tools.

**Start small:** SmolLM2 135M runs on CPU. Its answers are often imperfect; the goal
is to understand the pipeline. Allow roughly 1–2 GB working RAM and up to ~0.54 GB
for weights. The larger coding model needs substantially more memory.

Run cells in order. Downloads and training are disabled until you explicitly
change their switches. A fresh cache requires enabling the download cell before
loading. No generated code will be executed.


## 1. Prepare the environment

From a terminal at the repository root:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install torch --index-url https://download.pytorch.org/whl/cpu
python -m pip install -e '.[fine-tuning]'
python -m pip install jupyterlab ipykernel
python -m ipykernel install --prefix "$VIRTUAL_ENV" --name local-llms --display-name "Local LLMs"
python -m jupyterlab notebooks/local_llm_tutorial.ipynb
```

Select the **Local LLMs** kernel. The fine-tuning extra is only needed for the
optional LoRA section. For CUDA, use a compatible wheel from the PyTorch installer
instead of the CPU wheel. These commands install software, not model weights.

After updating repository Python code, use **Kernel → Restart Kernel and Run All Cells**
to discard previously imported functions. Reload this notebook from disk first if its
cells changed. An already open kernel does not automatically reload the loader.


In [1]:
from pathlib import Path
import importlib.metadata
import json
import os
import sys

# Accept a kernel launched from either the repository root or notebooks/.
root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "configs/models.yaml").is_file() and (p / "src/local_llms").is_dir()),
    None,
)
if root is None:
    raise RuntimeError("Start Jupyter from this repository or set its working directory there.")
os.chdir(root)
print("Repository:", root)
print("Kernel Python:", sys.executable)
for package in ("torch", "transformers", "local-llms"):
    print(package, importlib.metadata.version(package))


Repository: /home/kai/ros2_ws/src/local_llm
Kernel Python: /home/kai/ros2_ws/src/local_llm/.venv/bin/python
torch 2.14.0+cpu
transformers 5.17.0
local-llms 0.1.0


## 2. Choose a model and explicitly download it

The catalog stores a Hub repository, an immutable revision, and intended uses.
All bundled models have Apache-2.0 licenses; consult their model cards for terms.

- `smollm2-135m`: smallest general-purpose smoke model.
- `qwen2.5-0.5b`: a larger general chat baseline (~1 GB 16-bit weights).
- `qwen2.5-coder-1.5b`: stronger coding baseline (~3.1 GB 16-bit weights).

Use the same cache directory for downloading and loading. `None` uses the standard
Hugging Face cache. Public bundled models do not require authentication. For a gated
model, accept its terms and run `hf auth login` in a terminal; never put tokens here.


In [2]:
from local_llms.models import load_catalog, download_model, load_model

MODEL = "smollm2-135m"
CACHE_DIR = None
DEVICE = "cpu"   # Use "auto" or "cuda" for a compatible GPU.
DTYPE = "float32"  # Keeps the initial interpretability path unquantized.

catalog = load_catalog()
print(json.dumps(catalog, indent=2))


{
  "smollm2-135m": {
    "repo_id": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "revision": "12fd25f77366fa6b3b4b768ec3050bf629380bac",
    "purpose": [
      "general",
      "smoke-test"
    ],
    "license": "apache-2.0"
  },
  "qwen2.5-0.5b": {
    "repo_id": "Qwen/Qwen2.5-0.5B-Instruct",
    "revision": "7ae557604adf67be50417f59c2c2f167def9a775",
    "purpose": [
      "general"
    ],
    "license": "apache-2.0"
  },
  "qwen2.5-coder-1.5b": {
    "repo_id": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "revision": "2e1fd397ee46e1388853d2af2c993145b0f1098a",
    "purpose": [
      "coding",
      "general"
    ],
    "license": "apache-2.0"
  }
}


In [10]:
DOWNLOAD_MODEL = True  # Change to True to explicitly initiate this model's download.

if DOWNLOAD_MODEL:
    print(download_model(MODEL, cache_dir=CACHE_DIR))
else:
    print("Download skipped. Loading below requires this model revision to be cached.")


Fetching 13 files: 100%|██████████████████████████████| 13/13 [00:00<00:00, 1344.06it/s]

/home/kai/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM2-135M-Instruct/snapshots/12fd25f77366fa6b3b4b768ec3050bf629380bac


## 3. Load the ordinary PyTorch model and generate

Loading is offline-only. If the cache is missing or incomplete, run the download
cell above. The returned objects are the actual model and tokenizer, available for
your own experiments. Eager attention makes attention matrices accessible later.

Generation uses the model's chat template, greedy decoding, and a limit on **new**
tokens. Only the generated continuation is returned. Long prompts still consume
memory, and greedy decoding does not guarantee identical results across hardware.


In [11]:
import torch
from local_llms.inference import generate

model, tokenizer = load_model(
    MODEL, cache_dir=CACHE_DIR, device=DEVICE, dtype=DTYPE, attention="eager"
)
print(type(model).__name__)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print("Parameters:", format(parameter_count, ","))
print("Device:", model.device, "dtype:", next(model.parameters()).dtype)


ValueError: Model is not cached. Run: python scripts/download_model.py smollm2-135m

In [ ]:
answer = generate(
    model, tokenizer,
    "Why do trees help keep cities cool? Answer in two sentences.",
    system="You are a concise science tutor.",
    max_new_tokens=80,
)
print(answer)


## 4. Render an editable prompt and compare instructions

Prompt files use `$variable` placeholders. JSON braces stay literal; write `$$` for
a literal dollar sign. Missing variables raise an informative error.

Below we vary one instruction while keeping the model and source text fixed.
Inspect factual accuracy and instruction following rather than assuming the longer
response is better. This two-prompt comparison is a demonstration, not a benchmark.


In [ ]:
from local_llms.prompts import render_prompt

variables = json.loads(Path("prompts/examples/summary.json").read_text())
comparisons = []
for sentences in (1, 2):
    prompt = render_prompt("prompts/templates/summarize.txt", **(variables | {"sentences": sentences}))
    response = generate(model, tokenizer, prompt, max_new_tokens=80)
    comparisons.append({"sentences": sentences, "prompt": prompt, "response": response})
    print(f"\nRequested sentences: {sentences}\n{response}")


## 5. Ask for structured coding output

First validate a known response to understand the contract. Then ask the model to
produce that structure. JSON Schema checks fields and types; it cannot establish
whether an explanation, program, or test is correct.

SmolLM2 often fails this task. Failure is an expected experimental result. To try
the coding model, change `MODEL` above, explicitly download it, and restart/run the
notebook. Budget ~4–6 GB VRAM at 16-bit or 8+ GB RAM on CPU at float32.


In [ ]:
from local_llms.structured_output import parse_structured

schema = json.loads(Path("prompts/examples/coding.schema.json").read_text())
known_response = json.dumps({
    "explanation": "Multiply the input by itself.",
    "language": "python",
    "code": "def square(x):\n    return x * x",
    "tests": "assert square(3) == 9",
})
print(parse_structured(known_response, schema))


In [ ]:
coding_prompt = render_prompt(
    "prompts/templates/structured_coding.txt",
    task="Write a Python function square(x) that returns x squared.",
)
raw_response = generate(model, tokenizer, coding_prompt, max_new_tokens=256)
print("Raw response:\n", raw_response)
try:
    structured = parse_structured(raw_response, schema)
except ValueError as error:
    print("Validation failed:", error)
else:
    print("Validated fields:", list(structured))
    print(structured["code"])  # Display only; never eval/exec a model response.


## 6. Inspect a layer, hidden states, and attention

Find module names before choosing a hook. Bundled models expose `model.layers.0`;
other architectures can differ. A forward hook sees a module's output without
replacing the model. Remove it after the experiment so reruns do not accumulate hooks.

For this short input, a layer activation is typically `[batch, tokens, hidden]`.
Attention is typically `[batch, heads, query tokens, key tokens]`. Attention tensors
grow quadratically with sequence length, so keep inspection prompts short.


In [ ]:
module_names = [name for name, _ in model.named_modules()]
print("\n".join(module_names[:30]))
print("Architecture:", model.config.model_type)


In [ ]:
layer_name = "model.layers.0"
layer = dict(model.named_modules())[layer_name]
activations = {}

def capture_activation(module, inputs, output):
    tensor = output[0] if isinstance(output, tuple) else output
    activations[layer_name] = tensor.detach().cpu()

hook = layer.register_forward_hook(capture_activation)
try:
    inputs = tokenizer("The leaves absorb sunlight.", return_tensors="pt",
                       truncation=True, max_length=64).to(model.device)
    with torch.inference_mode():
        inspected = model(**inputs, output_hidden_states=True,
                          output_attentions=True, use_cache=False)
finally:
    hook.remove()

print("Captured:", layer_name, tuple(activations[layer_name].shape))
print("Hidden states:", [tuple(t.shape) for t in inspected.hidden_states])
print("Attention:", [tuple(t.shape) if t is not None else None
                     for t in (inspected.attentions or [])])
del inspected  # Release the full collection of retained states/attention matrices.


## 7. Optional: one LoRA training step

LoRA trains a small set of low-rank adapter weights while freezing the base model.
The existing example targets the query/value projections, masks the user prompt
from the training labels, and performs one supervised optimizer step. It then saves,
reloads, disables, and detaches the adapter. **One step checks the workflow; it does
not produce a useful fine-tuned model.**

Enable the switch below only after installing the `fine-tuning` extra. We release
the inference model before launching the script to avoid keeping two model copies
in memory. The subprocess uses this notebook's Python interpreter. A unique output
folder keeps reruns from overwriting an earlier adapter.


In [ ]:
RUN_LORA = False
adapter_dir = None

if RUN_LORA:
    import gc
    import subprocess
    import uuid

    del model, layer, inputs
    activations.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    adapter_dir = root / "outputs" / f"notebook-lora-{uuid.uuid4().hex[:8]}"
    command = [sys.executable, "examples/lora_example.py", MODEL,
               "--device", DEVICE, "--dtype", "float32", "--output", str(adapter_dir)]
    if CACHE_DIR is not None:
        command += ["--cache-dir", str(CACHE_DIR)]
    subprocess.run(command, check=True)
else:
    print("LoRA skipped. Set RUN_LORA=True to train and save a smoke-test adapter.")


In [ ]:
if adapter_dir is not None:
    model, tokenizer = load_model(
        MODEL, cache_dir=CACHE_DIR, device=DEVICE, dtype=DTYPE, adapter=str(adapter_dir)
    )
    question = "What is two plus two?"
    print("With adapter:", generate(model, tokenizer, question, max_new_tokens=32))
    with model.disable_adapter():
        print("Without adapter:", generate(model, tokenizer, question, max_new_tokens=32))
    model = model.unload()  # Detach without merging; restore the base model.


## 8. Keep a small experiment record

Record the revision, versions, prompts, hardware, and outputs alongside results.
The opt-in cell below writes your prompt comparison into ignored `outputs/`.
Review prompts before sharing them if they contain private data.


In [ ]:
SAVE_RESULTS = False

if SAVE_RESULTS:
    from datetime import datetime, timezone
    import uuid

    record = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "model": MODEL,
        "catalog_entry": catalog[MODEL],
        "device": DEVICE,
        "dtype": DTYPE,
        "versions": {p: importlib.metadata.version(p) for p in ("torch", "transformers")},
        "comparisons": comparisons,
    }
    destination = root / "outputs" / f"tutorial-{uuid.uuid4().hex[:8]}.json"
    destination.parent.mkdir(exist_ok=True)
    destination.write_text(json.dumps(record, indent=2), encoding="utf-8")
    print(destination)


## Where to go next

1. Add your own non-coding template and compare two phrasings on the same inputs.
2. Add a model in `configs/models.yaml`, recording its pinned revision and license.
3. Capture a different module and compare activations across two input sentences.
4. Replace the LoRA toy example with a licensed dataset, validation split, padding
   masks, checkpoints, and a task-specific evaluation before attempting longer training.

**Troubleshooting:** `ModuleNotFoundError` usually means the wrong kernel or an
uninstalled extra. A cache error needs an explicit download of the selected revision.
For out-of-memory errors, restart the kernel, choose the smallest model, and shorten
inputs. Quantization requires the optional bitsandbytes extra and CUDA in this
repository; preserve an unquantized path for interpretability. TransformerLens
integration is deferred; see [engineering notes](../notes.md).

See the [README](../README.md) for CLI equivalents, installation details, and limits.
